In [2]:
from pils.loader.stout import StoutLoader
from pils.loader.path import PathLoader
from pils.flight import Flight
from pils.sensors.camera import Camera, PhotogrammetryConfig

%matplotlib widget
import matplotlib.pyplot as plt


## 1 — Load flight metadata

Use `StoutLoader` if you have a Stout inventory, otherwise `PathLoader` to scan a directory.

The `flight_info` dict must contain at minimum:
- `drone_data_folder_path`
- `aux_data_folder_path`
- `proc_data_folder_path` — where `proc/photogrammetry/` will be created

In [3]:
# --- Option A: Stout inventory ---
# stout = StoutLoader()
# flight_meta = stout.load_single_flight(flight_name="flight_20251206_1304")

#--- Option B: directory scan ---
loader = PathLoader("/data/POLOCALC/")
flights = loader.load_single_flight(flight_name="flight_20251208_1506")
flight_meta = flights

flight_meta

2026-05-22 16:49:23,902 - pils.loader.path - INFO - Loading single flight: flight_id=None, flight_name=flight_20251208_1506
2026-05-22 16:49:23,903 - pils.loader.path - INFO - Loading all flights from all campaigns...
2026-05-22 16:49:23,903 - pils.loader.path - WARNING - Could not build flight dict for calibration  /data/POLOCALC/campaigns/202412/20241215/calibration : time data 'tion ' does not match format '%Y%m%d_%H%M'
2026-05-22 16:49:23,903 - pils.loader.path - WARNING - Could not build flight dict for Store-V2 /data/POLOCALC/campaigns/202412/.Spotlight-V100/Store-V2: time data '2' does not match format '%Y%m%d_%H%M'
2026-05-22 16:49:23,904 - pils.loader.path - WARNING - Could not build flight dict for 24_12_17 /data/POLOCALC/campaigns/202412/calibration_PUC/24_12_17: time data '7' does not match format '%Y%m%d_%H%M'
2026-05-22 16:49:23,904 - pils.loader.path - WARNING - Could not build flight dict for 24_12_18 /data/POLOCALC/campaigns/202412/calibration_PUC/24_12_18: time data '

{'campaign_name': '202511',
 'flight_name': 'flight_20251208_1506',
 'flight_date': '20251208',
 'takeoff_datetime': '2025-12-08T15:06:00+00:00',
 'landing_datetime': '2025-12-08T15:06:00+00:00',
 'drone_data_folder_path': '/data/POLOCALC/campaigns/202511/20251208/flight_20251208_1506/drone',
 'aux_data_folder_path': '/data/POLOCALC/campaigns/202511/20251208/flight_20251208_1506/aux',
 'processed_data_folder_path': '/data/POLOCALC/campaigns/202511/20251208/flight_20251208_1506/proc'}

## 1 — Load flight metadata

Use `StoutLoader` if you have a Stout inventory, otherwise `PathLoader` to scan a directory.

In [4]:
flight = Flight(flight_meta)

# Drone telemetry — auto-detects DJI / BlackSquare
flight.add_drone_data()

# Camera — use_photogrammetry=False loads raw frames/log (Alvium or Sony)
# This stores both:
#   flight.raw_data.payload_data.camera     → pl.DataFrame (for sync)
#   flight.raw_data.payload_data.camera_obj → Camera instance (for run_photogrammetry)
flight.add_camera_data(use_photogrammetry=False)

print(flight.raw_data)

2026-05-22 16:49:23,925 - pils.flight - INFO - Drone : /data/POLOCALC/campaigns/202511/20251208/flight_20251208_1506/drone/20251208_150555_drone.dat
2026-05-22 16:49:23,925 - pils.drones.DJIDrone - INFO - PATH: /data/POLOCALC/campaigns/202511/20251208/flight_20251208_1506/drone/20251208_150555_drone.dat
2026-05-22 16:49:26,184 - pils.drones.DJIDrone - INFO - False
2026-05-22 16:49:26,185 - pils.drones.DJIDrone - INFO - Tick unwrap at index 4023: 4,294,859,939 -> 827,284 (adding offset 2^32, total offset: 4,294,967,296)
2026-05-22 16:49:26,189 - pils.drones.DJIDrone - INFO - Loaded 4579 GPS messages from DAT file
2026-05-22 16:49:26,195 - pils.drones.DJIDrone - INFO - False
2026-05-22 16:49:26,196 - pils.drones.DJIDrone - INFO - Tick unwrap at index 3629: 4,294,709,539 -> 642,056 (adding offset 2^32, total offset: 4,294,967,296)
2026-05-22 16:49:26,196 - pils.drones.DJIDrone - INFO - Loaded 4146 RTK messages from DAT file
2026-05-22 16:49:26,285 - pils.drones.DJIDrone - INFO - Convertin

=== DRONE DATA ===
Drone:
shape: (8_584, 42)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ tick      ┆ msg_type ┆ GPS:date  ┆ GPS:time  ┆ … ┆ RTK:pos_f ┆ RTK:pos_f ┆ RTK:pos_f ┆ RTK:gps_s │
│ ---       ┆ ---      ┆ ---       ┆ ---       ┆   ┆ lg_3      ┆ lg_4      ┆ lg_5      ┆ tate      │
│ i64       ┆ i64      ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆          ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 90245775  ┆ 53234    ┆ null      ┆ null      ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ 90246203  ┆ 53234    ┆ null      ┆ null      ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ 90425531  ┆ 53234    ┆ null      ┆ null      ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ 91234448  ┆ 53234    ┆ null      ┆ null     

## 2 — Build Flight and load sensors

In [5]:
# photogrammetry_config.yaml lives in pils/config/
cfg = PhotogrammetryConfig("/home/fastori/Desktop/ARS/pils/pils/config/photogrammetryConfig.yaml")

print("Camera matrix:\n", cfg.camera_matrix)
print("Distortion coeffs:", cfg.distortion_coeffs)
print("Finder params:     ", cfg.finder)
print("PnP params:        ", cfg.pnp)
print("Drone corr params: ", cfg.drone_correlation)

Camera matrix:
 [[2.56960596e+03 0.00000000e+00 1.88156543e+03]
 [0.00000000e+00 2.56858496e+03 1.08713538e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion coeffs: [ 0.019473 -0.041976 -0.000273 -0.001083  0.030603]
Finder params:      {'box_height': 20, 'box_width': 20, 'color_tol': 10}
PnP params:         {'ransac': True}
Drone corr params:  {'trigger': False, 'payload': False, 'plot': False, 'use_fft': False}


## 3 — Load photogrammetry config

Edit `photogrammetry_config.yaml` with your camera calibration values before running.

In [6]:
import time
import cv2 as cv
import numpy as np

# ===================================================================
# Interactive Frame Viewer — Last 2 frames with click event handler
# ===================================================================

# Get the last 2 frames from camera data
camera_df = flight.raw_data.payload_data.camera
print(f"Total frames: {len(camera_df)}")

# Extract last 2 frame numbers
frame_nums = camera_df["frame"].to_list()[-2:]
print(f"Last 2 frames: {frame_nums}")

# Load actual frame images
camera_obj = flight.raw_data.payload_data.camera_obj
frames = []
for frame_num in frame_nums:
    try:
        frame = camera_obj._load_frame(frame_num)  # Adjust method name as needed
        frames.append(frame)
    except:
        # Fallback: try alternative loading
        pass

if not frames:
    print("[WARNING] Could not load frames. Skipping interactive viewer.")
else:
    # ===================================================================
    # Click event storage
    # ===================================================================
    
    click_assignments = {}  # frame_idx -> [(x, y), ...]
    for i, frame_num in enumerate(frame_nums):
        click_assignments[i] = []
    
    finished = [False]
    current_frame_idx = [0]
    
    # ===================================================================
    # Create figure
    # ===================================================================
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Display both frames
    for i, (ax, frame) in enumerate(zip(axes, frames)):
        rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        ax.imshow(rgb)
        ax.set_title(f"Frame {frame_nums[i]} — RIGHT click to mark | ENTER to finish", 
                     fontsize=11, color="cyan")
        ax.set_xlabel("x (pixels)")
        ax.set_ylabel("y (pixels)")
    
    point_markers = {0: [], 1: []}  # Markers per frame
    
    # ===================================================================
    # Mouse callback
    # ===================================================================
    
    def onclick(event):
        if event.inaxes is None or event.xdata is None or event.ydata is None:
            return
        
        # Right-click only (button 3)
        if event.button != 3:
            return
        
        # Find which axis was clicked
        frame_idx = None
        for i, ax in enumerate(axes):
            if event.inaxes == ax:
                frame_idx = i
                break
        
        if frame_idx is None:
            return
        
        x = int(event.xdata)
        y = int(event.ydata)
        
        # Store click
        click_assignments[frame_idx].append((x, y))
        
        # Draw marker
        marker, = axes[frame_idx].plot(x, y, 'r+', markersize=15, markeredgewidth=2)
        point_markers[frame_idx].append(marker)
        
        ax_title = f"Frame {frame_nums[frame_idx]} — {len(click_assignments[frame_idx])} click(s)"
        axes[frame_idx].set_title(ax_title, fontsize=11, color="lime")
        
        fig.canvas.draw_idle()
        print(f"[CLICK] Frame {frame_idx}, point {len(click_assignments[frame_idx])}: ({x}, {y})")
    
    # ===================================================================
    # Keyboard callback
    # ===================================================================
    
    def onkey(event):
        if event.key is None:
            return
        
        key = event.key.lower()
        
        # ENTER → finish
        if key in ("enter", "return"):
            finished[0] = True
            plt.close(fig)
            print("[DONE] Closed interactive viewer. Passing data to finder...")
        
        # Z → undo last click
        elif key == "z":
            for frame_idx in [0, 1]:
                if click_assignments[frame_idx]:
                    click_assignments[frame_idx].pop()
                    if point_markers[frame_idx]:
                        point_markers[frame_idx][-1].remove()
                        point_markers[frame_idx].pop()
                    ax_title = f"Frame {frame_nums[frame_idx]} — {len(click_assignments[frame_idx])} click(s)"
                    axes[frame_idx].set_title(ax_title, fontsize=11, color="orange")
            fig.canvas.draw_idle()
            print("[UNDO] Last click removed")
        
        # C → clear all
        elif key == "c":
            for frame_idx in [0, 1]:
                click_assignments[frame_idx] = []
                for marker in point_markers[frame_idx]:
                    marker.remove()
                point_markers[frame_idx] = []
                ax_title = f"Frame {frame_nums[frame_idx]} — 0 click(s)"
                axes[frame_idx].set_title(ax_title, fontsize=11, color="cyan")
            fig.canvas.draw_idle()
            print("[CLEAR] All clicks removed")
    
    # ===================================================================
    # Connect callbacks
    # ===================================================================
    
    cid_click = fig.canvas.mpl_connect("button_press_event", onclick)
    cid_key = fig.canvas.mpl_connect("key_press_event", onkey)
    
    plt.tight_layout()
    plt.show()
    
    # ===================================================================
    # BLOCK EXECUTION until ENTER is pressed
    # ===================================================================
    
    print("[WAITING] Press ENTER in the plot window to continue...")
    while not finished[0]:
        plt.pause(0.05)
        time.sleep(0.05)
    
    print(f"\n✅ Frame 0 clicks: {click_assignments[0]}")
    print(f"✅ Frame 1 clicks: {click_assignments[1]}")


Total frames: 1514074


ColumnNotFoundError: "frame" not found

In [ ]:
# ===================================================================
# Pass clicked coordinates to finder workflow (CHECKPOINT)
# ===================================================================

print("\n" + "="*70)
print("MANUAL QUALITY CONTROL CHECKPOINT")
print("="*70)

# Summary of manual corrections
manual_corrections = {}
total_points = 0

for frame_idx in range(len(frame_nums)):
    frame_num = frame_nums[frame_idx]
    points = click_assignments[frame_idx]
    manual_corrections[frame_num] = points
    total_points += len(points)
    
    if points:
        print(f"\n✅ Frame {frame_num}:")
        print(f"   Manual corrections: {len(points)} point(s)")
        for i, (x, y) in enumerate(points):
            print(f"     Correction [{i}]: pixel ({x}, {y})")
    else:
        print(f"\n⚠️  Frame {frame_num}: No manual corrections")

print(f"\nTotal manual corrections collected: {total_points} point(s)")

if total_points == 0:
    print("[INFO] No manual corrections — will proceed with automatic finder")
else:
    print("[INFO] Finder will now use these as anchor points for better detection across ALL frames")

# Store for use in photogrammetry pipeline
frame_click_data = manual_corrections

print("\n" + "="*70)
print("✅ CHECKPOINT COMPLETE — Ready for full photogrammetry pipeline")
print("="*70 + "\n")


## 4 — Run the photogrammetry pipeline

- `csv_file`: geodetic targets + telescope positions CSV
- `output_dir`: where intermediate `.ecsv` dictionaries are saved
- `check_results`: if set, diagnostic plots are saved here
- `start_from_dict`: resume mid-pipeline by pointing to an existing `.ecsv`

In [ ]:
# ===================================================================
# FULL PHOTOGRAMMETRY PIPELINE (Step 4)
# Runs on ALL frames, using manual corrections as anchor points
# ===================================================================

print("\n" + "="*70)
print("INITIATING FULL PHOTOGRAMMETRY PIPELINE")
print("="*70)

print("\n[INFO] Manual corrections from click event:")
for frame_num, points in frame_click_data.items():
    if points:
        print(f"  Frame {frame_num}: {len(points)} manual anchor point(s)")

# Grab the Camera object stored by add_camera_data()
camera_obj = flight.raw_data.payload_data.camera_obj

print("\n[PIPELINE STAGES]:")
print("  → p1: Load & sync frames across all timestamps")
print("  → p2: Find targets (informed by manual anchor points)")
print("  → p3: Solve PnP camera attitude (rvec, tvec)")
print("  → p4: Correlate with drone GPS telemetry")
print("  → p5: Apply fusion corrections (GPS + camera)")
print("  → p6: Transform to telescope reference frame")
print("  → p7: Compute line-of-sight (LOS) frame attitude")
print("\n[RUNNING]...\n")

result = camera_obj.run_photogrammetry(
    csv_file="/data/POLOCALC/campaigns/202511/metadata/202511_coordinates.csv",
    config=cfg,
    flight=flight,
    output_dir="/home/fastori/Desktop/ARS/photogrammetry_results_1225",
    check_results="/home/fastori/Desktop/ARS/photogrammetry_results_1225",  # Save diagnostic plots
    start_from_dict=None,
)

print(f"\n[SUCCESS] Pipeline complete!")
print(f"Result: {result.shape[0]} frames × {result.shape[1]} columns\n")

result.head()


## 5 — Inspect the result

### 5.1 Column overview

In [ ]:
# Group columns by pipeline step for easier orientation
groups = {
    "p2 — image targets":     [c for c in result.columns if c in ["frame", "x", "y", "target_id"]],
    "p3 — PnP attitude":      [c for c in result.columns if c.startswith(("rvec", "tvec", "quat", "proj"))],
    "p4 — drone GPS":         [c for c in result.columns if c.startswith(("drone_", "time"))],
    "p5 — GPS-corrected":     [c for c in result.columns if c.endswith("_corr") or "projection_error" in c],
    "p6 — telescope frame":   [c for c in result.columns if c in ["tel_name", "az", "el"]],
    "p7 — LOS frame":         [c for c in result.columns if "LOS" in c],
}

for group, cols in groups.items():
    if cols:
        print(f"\n{group}")
        print("  ", cols)

### 5.2 Projection error — quality check

Low projection error (< 2–3 px) means the PnP solution is reliable for that frame.

In [ ]:
pe = result.select(["frame", "time", "projection_error"]).drop_nulls()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pe["time"].to_numpy(), pe["projection_error"].to_numpy(), lw=1)
ax.axhline(2.0, color="orange", ls="--", label="2 px threshold")
ax.axhline(5.0, color="red",    ls="--", label="5 px threshold")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Projection error (px)")
ax.set_title("PnP projection error per frame")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Median: {pe['projection_error'].median():.2f} px")
print(f"Max:    {pe['projection_error'].max():.2f} px")
print(f"Frames > 5px: {(pe['projection_error'] > 5).sum()}")

### 5.3 Attitude — roll, pitch, yaw over time

In [ ]:
import numpy as np

# Use GPS-corrected rvec if available, fall back to raw PnP
rvec_cols = ["rvec_x_corr", "rvec_y_corr", "rvec_z_corr"]
if not all(c in result.columns for c in rvec_cols):
    rvec_cols = ["rvec_x", "rvec_y", "rvec_z"]

att = result.select(["time"] + rvec_cols).drop_nulls()
t   = att["time"].to_numpy()
rv  = att.select(rvec_cols).to_numpy()  # (N, 3) Rodrigues vectors

# Convert to degrees for readability
angles_deg = np.degrees(rv)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
labels = ["Roll (°)", "Pitch (°)", "Yaw (°)"]
for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.plot(t, angles_deg[:, i], lw=1)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (s)")
axes[0].set_title("Camera attitude (GPS-corrected rvec)")
plt.tight_layout()
plt.show()

### 5.4 Drone position in ENU

In [ ]:
pos = result.select(["time", "drone_E", "drone_N", "drone_U"]).drop_nulls()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground track
axes[0].plot(pos["drone_E"].to_numpy(), pos["drone_N"].to_numpy(), lw=1)
axes[0].set_xlabel("East (m)")
axes[0].set_ylabel("North (m)")
axes[0].set_title("Ground track (ENU)")
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.3)

# Altitude over time
axes[1].plot(pos["time"].to_numpy(), pos["drone_U"].to_numpy(), lw=1, color="steelblue")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Up / altitude (m)")
axes[1].set_title("Altitude over time")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.5 LOS frame attitude (p7 output)

In [ ]:
los_cols = ["time", "yaw_LOS", "pitch_LOS", "roll_LOS"]
if all(c in result.columns for c in los_cols):
    los = result.select(los_cols).drop_nulls()
    t   = los["time"].to_numpy()

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for ax, col, label in zip(
        axes,
        ["yaw_LOS", "pitch_LOS", "roll_LOS"],
        ["Yaw LOS (°)", "Pitch LOS (°)", "Roll LOS (°)"],
    ):
        ax.plot(t, np.degrees(los[col].to_numpy()), lw=1)
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("Time (s)")
    axes[0].set_title("Drone attitude in line-of-sight frame")
    plt.tight_layout()
    plt.show()
else:
    print("LOS columns not found — pipeline may not have reached step p7.")

## 6 — Save result

Save as Parquet for fast reloading, or as CSV for external tools.

In [ ]:
# The result is already saved automatically to:
#   flight.flight_info['proc_data_folder_path'] / 'photogrammetry' / 'attitude_reconstruction.parquet'
#
# Reload it later with:
import os
proc_path = flight.flight_info["proc_data_folder_path"]
parquet_path = os.path.join(proc_path, "photogrammetry", "attitude_reconstruction.parquet")

import polars as pl
result = pl.read_parquet(parquet_path)
print(f"Loaded {result.shape[0]} frames from {parquet_path}")
result.head()

## 7 — Resume from a mid-pipeline checkpoint

If the pipeline failed or you want to re-run from step p4 onwards, point `start_from_dict` at the last good dictionary.

In [ ]:
# Resume from a mid-pipeline checkpoint — skips all steps before p4
import os
proc_path = flight.flight_info["proc_data_folder_path"]
checkpoint = os.path.join(proc_path, "photogrammetry", "dictionary_p3.ecsv")

result = camera_obj.run_photogrammetry(
    csv_file="/path/to/targets.csv",
    config=cfg,
    flight=flight,
    start_from_dict=checkpoint,
)

result.head()